# 第 7 天：咖啡元胞自动机

> Aaronson、Carroll、Ouellette（2014）— [arXiv:1405.6903](https://arxiv.org/abs/1405.6903)

本 Notebook 演示论文的关键实验：
1. 构建咖啡元胞自动机（二值网格，随机交换相邻格点）
2. 测量表观复杂度（粗粒化状态的 gzip 压缩大小）
3. 展示复杂度先升后降，而熵单调增加
4. 比较有相互作用与无相互作用模型
5. 复现调整后的粗粒化（第 6 节）


In [ ]:
import numpy as np
import gzip
import matplotlib.pyplot as plt
%matplotlib inline

from implementation import (
    CoffeeAutomaton, run_simulation, compare_models,
    coarse_grain, threshold_array, row_majority_adjust,
    measure_entropy, measure_complexity, gzip_size, grid_to_bytes
)
print('导入成功')

## 1. 咖啡元胞自动机模型

第 3.1 节：N x N 二值网格，上半部 = 奶油（1），下半部 = 咖啡（0）。
每步随机选择一对相邻且不同的格点，交换它们。


In [ ]:
# 创建 50x50 的有相互作用自动机
ca = CoffeeAutomaton(grid_size=50, model='interacting', seed=42)

# 可视化初始状态
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
states = [0, 5000, 25000, 50000]

for i, t in enumerate(states):
    if t > ca.time:
        ca.step(t - ca.time)
    axes[i].imshow(ca.grid, cmap='gray_r', vmin=0, vmax=1)
    axes[i].set_title(f't = {t}')
    axes[i].axis('off')

plt.suptitle('咖啡元胞自动机：有相互作用模型（50x50）')
plt.tight_layout()
plt.show()

## 2. 测量表观复杂度

论文的关键测量（第 5 节）：
- **熵：** gzip（细粒度状态），作为 K(x) 的代理量
- **复杂度：** gzip（粗粒化状态），作为 K(f(x)) 的代理量

粗粒化：在 g x g 块内取平均，按阈值分箱，再压缩。


In [ ]:
# 运行完整模拟并收集测量结果
print('正在运行模拟（50x50，80000 步）……')
results = run_simulation(
    grid_size=50, total_steps=80000, num_snapshots=80,
    model='interacting', seed=42
)

# 绘制熵与复杂度
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(results['times'], results['entropy'], 'b-')
ax1.set_xlabel('时间步')
ax1.set_ylabel('熵（gzip 字节数）')
ax1.set_title('熵：单调增加')

ax2.plot(results['times'], results['complexity'], 'r-')
peak_idx = np.argmax(results['complexity'])
ax2.axvline(results['times'][peak_idx], color='gray', ls='--', alpha=0.5)
ax2.set_xlabel('时间步')
ax2.set_ylabel('复杂度（gzip 字节数）')
ax2.set_title('表观复杂度：先升后降')

plt.tight_layout()
plt.show()
print(f'复杂度峰值位于 t = {results["times"][peak_idx]}')

## 3. 粗粒化可视化

在复杂度峰值处比较细粒度与粗粒化状态。
这对应论文图 3 和图 11。


In [ ]:
# 显示不同时刻的细粒度与粗粒化状态
if 'grids' in results:
    indices = [0, peak_idx, len(results['grids'])-1]
    labels = ['t=0（初始）', f't={results["times"][peak_idx]}（峰值）', f't={results["times"][-1]}（结束）']
    
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    for col, (idx, label) in enumerate(zip(indices, labels)):
        grid = results['grids'][idx]
        axes[0, col].imshow(grid, cmap='gray_r', vmin=0, vmax=1)
        axes[0, col].set_title(label)
        axes[0, col].axis('off')
        
        coarse = coarse_grain(grid, grain_size=5)
        discrete = threshold_array(coarse, 7)
        discrete = row_majority_adjust(discrete)
        axes[1, col].imshow(discrete, cmap='gray_r')
        axes[1, col].axis('off')
    
    axes[0, 0].set_ylabel('细粒度', fontsize=11)
    axes[1, 0].set_ylabel('粗粒化', fontsize=11)
    plt.suptitle('细粒度与粗粒化状态')
    plt.tight_layout()
    plt.show()

## 4. 有相互作用与无相互作用

论文最重要的发现（第 6.2 节）：真正的复杂性需要相互作用。
无相互作用粒子始终不会产生宏观结构，因为每个粒子都在独立随机游走。


In [ ]:
# 比较两种模型
inter, non_inter = compare_models(
    grid_size=40, total_steps=50000, num_snapshots=60, seed=42
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(inter['times'], inter['complexity'], 'r-', label='有相互作用')
ax1.plot(non_inter['times'], non_inter['complexity'], 'b--', label='无相互作用')
ax1.set_xlabel('时间步')
ax1.set_ylabel('复杂度（字节）')
ax1.set_title('复杂度：有相互作用与无相互作用')
ax1.legend()

ax2.plot(inter['times'], inter['entropy'], 'r-', label='有相互作用')
ax2.plot(non_inter['times'], non_inter['entropy'], 'b--', label='无相互作用')
ax2.set_xlabel('时间步')
ax2.set_ylabel('熵（字节）')
ax2.set_title('两种模型的熵')
ax2.legend()

plt.tight_layout()
plt.show()

## 5. 第 5 节与第 6 节：阈值化方法

第 5 节使用 3 分箱阈值化，会产生边界像素伪影。
第 6 节通过 7 分箱与行众数调整修正这个问题。


In [ ]:
# 比较 3 分箱（第 5 节）与调整后的 7 分箱（第 6 节）
results_s5 = run_simulation(
    grid_size=50, total_steps=80000, num_snapshots=80,
    model='interacting', num_buckets=3, adjust=False, seed=42
)
results_s6 = run_simulation(
    grid_size=50, total_steps=80000, num_snapshots=80,
    model='interacting', num_buckets=7, adjust=True, seed=42
)

plt.figure(figsize=(10, 5))
plt.plot(results_s5['times'], results_s5['complexity'], 'r-',
         label='3 分箱，不调整（第 5 节）', alpha=0.7)
plt.plot(results_s6['times'], results_s6['complexity'], 'b-',
         label='7 分箱，调整（第 6 节）', alpha=0.7)
plt.xlabel('时间步')
plt.ylabel('表观复杂度（字节）')
plt.title('粗粒化方法比较')
plt.legend()
plt.tight_layout()
plt.show()

## 总结

以上复现了论文的关键结果：

1. 表观复杂度（粗粒化状态的 gzip 压缩大小）先升后降
2. 熵（细粒度状态的 gzip 压缩大小）单调增加
3. 先升后降需要粒子间的相互作用
4. 调整后的粗粒化（第 6 节）消除阈值化伪影

开放问题：解析证明有相互作用模型必然变复杂。
